In [ ]:
# @title 👗 OutfitAura GPU Worker Setup
import os

# 1. Enter your Ngrok token here
NGROK_TOKEN = "Put your ngrok auth token" # Get from https://dashboard.ngrok.com/get-started/your-authtoken

print("Installing dependencies...")
!pip install -q fastapi uvicorn python-multipart diffusers transformers accelerate safetensors pyngrok huggingface_hub opencv-python scipy

# 2. Download the CatVTON trainable weights (if you don't have them locally in Colab)
from huggingface_hub import hf_hub_download
import torch

os.makedirs("checkpoints/catvton", exist_ok=True)
weights_path = "checkpoints/catvton/CatVTON/trainable_weights.pt"

if not os.path.exists(weights_path):
    print("Downloading CatVTON weights...")
    hf_hub_download(repo_id="zhengchong/CatVTON", filename="CatVTON/catvton_pipeline.pt", local_dir="checkpoints/catvton")
    # Rename to match your tryon_service.py expectation
    os.rename("checkpoints/catvton/CatVTON/catvton_pipeline.pt", weights_path)

# 3. Create a local copy of your tryon_service.py logic
print("Setting up Try-On Service...")
with open("colab_worker.py", "w") as f:
    f.write('''
import os
import base64
import torch
from io import BytesIO
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
from PIL import Image
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
import numpy as np
from torchvision import transforms
import cv2
import scipy.ndimage as ndimage

app = FastAPI()

# Minimal implementation of your CatVTONService for Colab
class ColabCatVTON:
    def __init__(self):
        self.device = torch.device("cuda")
        self.base_model = "stable-diffusion-v1-5/stable-diffusion-inpainting"

        print("Loading models to GPU...")
        self.vae = AutoencoderKL.from_pretrained(self.base_model, subfolder="vae", torch_dtype=torch.float32).to(self.device)
        self.unet = UNet2DConditionModel.from_pretrained(self.base_model, subfolder="unet", torch_dtype=torch.float32).to(self.device)
        self.noise_scheduler = DDPMScheduler.from_pretrained(self.base_model, subfolder="scheduler")

        weights_path = "checkpoints/catvton/trainable_weights.pt"
        checkpoint = torch.load(weights_path, map_location="cpu", weights_only=False)
        trainable_state = checkpoint.get("trainable_state_dict", checkpoint)

        # Depending on the downloaded checkpoint structure, adjust keys:
        if "unet" in trainable_state: trainable_state = trainable_state["unet"]

        current_state = self.unet.state_dict()
        loaded = 0
        for name, param in trainable_state.items():
            if name in current_state:
                current_state[name].copy_(param)
                loaded += 1

        print(f"Loaded {loaded} custom weights.")
        self.vae.eval()
        self.unet.eval()
        self.uncond_embeddings = torch.zeros((1, 77, 768), device=self.device, dtype=torch.float32)

        self.normalize_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])

    def generate_tryon(self, person_img, cloth_img, parse_img):
        # Resize inputs (using default 512x384 for T4 GPU)
        pil_resize = (384, 512)

        # Basic mask generation (simplified version of your logic)
        parse_array = np.array(parse_img.resize(pil_resize, Image.NEAREST))
        mask_labels = [5, 6, 7, 11, 15, 16, 21, 22]
        binary_mask = np.isin(parse_array, mask_labels)
        filled_binary_mask = ndimage.binary_fill_holes(binary_mask)
        mask = filled_binary_mask.astype(np.uint8) * 255
        mask = cv2.dilate(mask, np.ones((5, 5), np.uint8), iterations=2)

        person_array = np.array(person_img.resize(pil_resize, Image.BILINEAR))
        agnostic_array = person_array.copy()
        agnostic_array[mask > 0] = 0

        agnostic_img = Image.fromarray(agnostic_array)
        mask_img = Image.fromarray(mask)
        cloth_img = cloth_img.resize(pil_resize, Image.BILINEAR)

        cloth = self.normalize_transform(cloth_img).unsqueeze(0).to(self.device)
        agnostic = self.normalize_transform(agnostic_img).unsqueeze(0).to(self.device)
        mask_t = transforms.ToTensor()(mask_img).unsqueeze(0).to(self.device)

        combined_input = torch.cat([agnostic, cloth], dim=3)
        combined_mask = torch.cat([mask_t, torch.zeros_like(mask_t)], dim=3)

        with torch.no_grad():
            latents = self.vae.encode(combined_input).latent_dist.sample() * self.vae.config.scaling_factor
            mask_latent = torch.nn.functional.interpolate(combined_mask, size=latents.shape[2:], mode="nearest")

            # Simplified inference loop
            self.noise_scheduler.set_timesteps(50, device=self.device)
            noise_latents = torch.randn_like(latents, device=self.device)
            cond_embeddings = self.uncond_embeddings.repeat(1, 1, 1)

            # Create unconditional latents (zeros for cloth part)
            uncond_latents = latents.clone()
            garment_width = latents.shape[3] // 2
            uncond_latents[:, :, :, garment_width:] = torch.randn_like(uncond_latents[:, :, :, garment_width:])
            uncond_mask_latent = torch.zeros_like(mask_latent)

            for t in self.noise_scheduler.timesteps:
                model_input = torch.cat([
                    torch.cat([noise_latents, mask_latent, latents], dim=1),
                    torch.cat([noise_latents, uncond_mask_latent, uncond_latents], dim=1)
                ], dim=0)

                embeddings = torch.cat([cond_embeddings, cond_embeddings], dim=0)
                noise_pred = self.unet(model_input, torch.cat([t.unsqueeze(0), t.unsqueeze(0)]), encoder_hidden_states=embeddings).sample

                noise_pred_cond, noise_pred_uncond = noise_pred.chunk(2, dim=0)
                noise_pred = noise_pred_uncond + 2.0 * (noise_pred_cond - noise_pred_uncond)
                noise_latents = self.noise_scheduler.step(noise_pred, t, noise_latents).prev_sample

            decoded = self.vae.decode(noise_latents / self.vae.config.scaling_factor).sample
            result = torch.clamp((decoded[:, :, :, :garment_width] + 1) / 2.0, 0, 1)
            tryon_np = (result[0].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)

        return Image.fromarray(tryon_np)

print("Initializing Try-On model...")
service = ColabCatVTON()
print("Try-On model ready!")

@app.post("/")
async def process_tryon(
    person_image: UploadFile = File(...),
    garment_image: UploadFile = File(...),
    parsing_image: UploadFile = File(...)
):
    try:
        p_img = Image.open(person_image.file).convert("RGB")
        g_img = Image.open(garment_image.file).convert("RGB")
        mask_img = Image.open(parsing_image.file).convert("L")

        result_img = service.generate_tryon(p_img, g_img, mask_img)

        buffered = BytesIO()
        result_img.save(buffered, format="PNG")
        img_str = base64.b64encode(buffered.getvalue()).decode()
        return {"status": "success", "tryon_image_base64": img_str}
    except Exception as e:
        import traceback
        traceback.print_exc()
        return JSONResponse(status_code=500, content={"status": "error", "error": str(e)})
''')

# 4. Start the server and create Ngrok tunnel
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import threading

# Authenticate ngrok
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000).public_url

print(f"\n{'='*50}")
print(f"🚀 GPU Worker is LIVE!")
print(f"🔗 Copy this URL into your HF Space Variables:")
print(f"   COLAB_TRYON_URL = {public_url}")
print(f"{'='*50}\n")

# Run the server in a separate thread
def run_uvicorn():
    uvicorn.run("colab_worker:app", host="0.0.0.0", port=8000)

nest_asyncio.apply()
uvicorn_thread = threading.Thread(target=run_uvicorn)
uvicorn_thread.start()

Installing dependencies...
Setting up Try-On Service...

🚀 GPU Worker is LIVE!
🔗 Copy this URL into your HF Space Variables:
   COLAB_TRYON_URL = https://7a59-34-125-100-162.ngrok-free.app

